# Fixed-design distortion additivity confirmation

This notebook runs the additivity test for seeds 42, 43, and 44 in one Colab session.

All seeds use the same selected layers, the same layer pairs, and the same layer groups. Each seed uses 1,024 calibration examples and 1,000 test examples. The notebook reuses the existing dense checkpoints and does not retrain models.

## Requirements

- GPU runtime enabled.
- Colab Secret GITHUB_TOKEN with read access to the private my-thesis repository and its private submodules.
- Optional Colab Secret GITHUB_WRITE_TOKEN with Contents read/write access to rdhp-rdp-colab-experiment.
- Dense checkpoints already available for seeds 42, 43, and 44.
- CIFAR-10 extracted at /content/drive/MyDrive/cifar10/cifar-10-batches-py.

The final cell is disabled by default. Set PUBLISH_RESULTS = True only when you want to upload the results.

In [ ]:
from pathlib import Path
import copy
import getpass
import json
import os
import subprocess
import sys
import time
from argparse import Namespace

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from google.colab import drive

OWNER = 'hamidrg20001379'
REPO_NAME = 'my-thesis'
REPO_URL = f'https://github.com/{OWNER}/{REPO_NAME}.git'
REPO = Path('/content') / REPO_NAME

if REPO.exists() and not (REPO / '.git').exists():
    raise RuntimeError(f'{REPO} exists but is not a Git checkout.')

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass
if not token:
    token = getpass.getpass('GitHub token with private-repository read access: ')
if not token:
    raise RuntimeError('A GitHub token is required.')

askpass = Path('/tmp/github-askpass-all-seeds.sh')
askpass.write_text(
    '#!/bin/sh\n'
    'case "$1" in\n'
    "  *Username*) printf '%s\n' x-access-token ;;\n"
    "  *Password*) printf '%s\n' \"$GITHUB_TOKEN_FOR_ASKPASS\" ;;\n"
    "  *) printf '\n' ;;\n"
    'esac\n'
)
askpass.chmod(0o700)
git_env = os.environ.copy()
git_env.update({
    'GIT_ASKPASS': str(askpass),
    'GIT_TERMINAL_PROMPT': '0',
    'GITHUB_TOKEN_FOR_ASKPASS': token,
})
try:
    if not REPO.exists():
        subprocess.run(
            ['git', 'clone', '--recurse-submodules', REPO_URL, str(REPO)],
            env=git_env, check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(REPO), 'submodule', 'update', '--init', '--recursive'],
            env=git_env, check=True,
        )
finally:
    askpass.unlink(missing_ok=True)

EXPERIMENTS = REPO / 'working' / 'experiments'
sys.path.insert(0, str(EXPERIMENTS))

from rdp_resnet32_adapter import (
    load_resnet32,
    logits,
    make_loaders,
    mask_topk,
    prunable_layers,
    seed_everything,
)

SEEDS = [42, 43, 44]
SELECTION_SEED = 0
TRAIN_SAMPLES = 5000
CALIB_SAMPLES = 1024
TEST_SAMPLES = 1000
BATCH_SIZE = 128
LAYER_COUNT = 12
PAIR_COUNT = 24
GROUP_COUNT = 12
GROUP_SIZE = 4
RATIOS = [0.25, 0.50, 0.75]

DRIVE_ROOT = Path('/content/drive/MyDrive')
if not DRIVE_ROOT.exists():
    drive.mount('/content/drive')

DATA_ROOT = DRIVE_ROOT / 'cifar10'
AGG_ROOT = DRIVE_ROOT / 'rdhp_rdp_additivity_fixed_all_seeds'
AGG_ROOT.mkdir(parents=True, exist_ok=True)

dataset_dir = DATA_ROOT / 'cifar-10-batches-py'
required = [dataset_dir / name for name in [
    'data_batch_1', 'data_batch_2', 'data_batch_3',
    'data_batch_4', 'data_batch_5', 'test_batch',
]]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Incomplete CIFAR-10 dataset: ' + ', '.join(missing))

print('Seeds:', SEEDS)
print('Aggregate output:', AGG_ROOT)

In [ ]:
probe = load_resnet32()
layer_names = [
    name for name, module in probe.named_modules()
    if isinstance(module, (torch.nn.Conv2d, torch.nn.Linear))
]
del probe

selected_layers = np.linspace(
    0, len(layer_names) - 1, min(LAYER_COUNT, len(layer_names)), dtype=int
).tolist()
selected_layers = list(dict.fromkeys(int(i) for i in selected_layers))

rng = np.random.default_rng(SELECTION_SEED)
pair_candidates = [
    (int(a), int(b))
    for position, a in enumerate(selected_layers)
    for b in selected_layers[position + 1:]
]
pair_indices = [
    pair_candidates[i]
    for i in rng.choice(
        len(pair_candidates),
        size=min(PAIR_COUNT, len(pair_candidates)),
        replace=False,
    )
]

group_indices = []
for _ in range(min(GROUP_COUNT, 100)):
    group_indices.append(
        rng.choice(
            selected_layers,
            size=min(GROUP_SIZE, len(selected_layers)),
            replace=False,
        ).tolist()
    )

print('Prunable layers:', len(layer_names))
print('Fixed selected layers:', [(i, layer_names[i]) for i in selected_layers])
print('Fixed pairs:', pair_indices)
print('Fixed groups:', group_indices)

In [ ]:
def summarize(df, prefix):
    return {
        f'{prefix}_rows': int(len(df)),
        f'{prefix}_calib_mean_ratio_median': float(df['calib_mean_ratio'].median()),
        f'{prefix}_calib_mean_relative_interaction_median': float(
            (df['calib_mean_interaction'] /
             df['calib_mean_sum'].clip(lower=1e-12)).median()
        ),
        f'{prefix}_test_mean_ratio_median': float(df['test_mean_ratio'].median()),
        f'{prefix}_test_mean_relative_interaction_median': float(
            (df['test_mean_interaction'] /
             df['test_mean_sum'].clip(lower=1e-12)).median()
        ),
        f'{prefix}_calib_max_ratio_median': float(df['calib_max_ratio'].median()),
        f'{prefix}_test_max_ratio_median': float(df['test_max_ratio'].median()),
    }

def run_seed(seed):
    seed_everything(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    run_root = DRIVE_ROOT / f'rdhp_rdp_runs_seed_{seed}'
    checkpoint = run_root / 'dense_resnet32.pt'
    output_root = run_root / 'distortion_additivity_fixed'
    output_root.mkdir(parents=True, exist_ok=True)

    if not checkpoint.exists():
        raise FileNotFoundError(f'Missing checkpoint for seed {seed}: {checkpoint}')

    loader_args = Namespace(
        synthetic=False,
        data_root=str(DATA_ROOT),
        download=False,
        train_samples=TRAIN_SAMPLES,
        calib_samples=CALIB_SAMPLES,
        test_samples=TEST_SAMPLES,
        batch_size=BATCH_SIZE,
        seed=seed,
    )
    (train_loader, calibration_loader, test_loader), split_info = make_loaders(loader_args)

    payload = torch.load(checkpoint, map_location=device, weights_only=False)
    state_dict = payload['model_state_dict'] if 'model_state_dict' in payload else payload
    model = load_resnet32().to(device)
    model.load_state_dict(state_dict)
    model.eval()

    layers = prunable_layers(model)
    if len(layers) != len(layer_names):
        raise RuntimeError('Layer count differs from the fixed design.')

    dense_calib = logits(model, calibration_loader, device)
    dense_test = logits(model, test_loader, device)

    def all_one_masks():
        return [torch.ones_like(layer.weight, dtype=torch.bool) for layer in layers]

    def masks_for(indices_and_ratios):
        masks = all_one_masks()
        for layer_index, ratio in indices_and_ratios:
            masks[layer_index] = mask_topk(
                layers[layer_index].weight, float(ratio)
            ).detach().cpu()
        return masks

    def measure_masks(masks):
        masked = copy.deepcopy(model).to(device)
        with torch.no_grad():
            for layer, mask in zip(prunable_layers(masked), masks):
                layer.weight.mul_(mask.to(device=device, dtype=layer.weight.dtype))
        masked.eval()
        masked_calib = logits(masked, calibration_loader, device)
        masked_test = logits(masked, test_loader, device)
        calib_error = (masked_calib - dense_calib).flatten(1).square().mean(dim=1)
        test_error = (masked_test - dense_test).flatten(1).square().mean(dim=1)
        result = {
            'calib_mean': float(calib_error.mean()),
            'calib_max': float(calib_error.max()),
            'test_mean': float(test_error.mean()),
            'test_max': float(test_error.max()),
        }
        del masked, masked_calib, masked_test
        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return result

    single_lookup = {}
    single_rows = []
    started = time.monotonic()

    for layer_index in selected_layers:
        for ratio in RATIOS:
            values = measure_masks(masks_for([(layer_index, ratio)]))
            row = {
                'layer_index': layer_index,
                'layer_name': layer_names[layer_index],
                'ratio': ratio,
                **values,
            }
            single_rows.append(row)
            single_lookup[(layer_index, ratio)] = row

    single_df = pd.DataFrame(single_rows)
    pair_rows = []
    for layer_a, layer_b in pair_indices:
        for ratio in RATIOS:
            values = measure_masks(masks_for([(layer_a, ratio), (layer_b, ratio)]))
            a = single_lookup[(layer_a, ratio)]
            b = single_lookup[(layer_b, ratio)]
            for metric in ('calib_mean', 'calib_max', 'test_mean', 'test_max'):
                joint = values[metric]
                additive = a[metric] + b[metric]
                values[f'{metric}_sum'] = additive
                values[f'{metric}_interaction'] = joint - additive
                values[f'{metric}_ratio'] = joint / max(additive, 1e-12)
            pair_rows.append({
                'layer_a': layer_a,
                'layer_a_name': layer_names[layer_a],
                'layer_b': layer_b,
                'layer_b_name': layer_names[layer_b],
                'ratio': ratio,
                **values,
            })
    pair_df = pd.DataFrame(pair_rows)

    group_rows = []
    group_ratio = 0.50
    for group_id, group in enumerate(group_indices):
        values = measure_masks(masks_for([(int(i), group_ratio) for i in group]))
        row = {
            'group_id': group_id,
            'layer_indices': ','.join(str(int(i)) for i in group),
            'layer_names': ' | '.join(layer_names[int(i)] for i in group),
            'ratio': group_ratio,
            **values,
        }
        for metric in ('calib_mean', 'calib_max', 'test_mean', 'test_max'):
            additive = sum(
                single_lookup[(int(i), group_ratio)][metric] for i in group
            )
            row[f'{metric}_sum'] = additive
            row[f'{metric}_interaction'] = values[metric] - additive
            row[f'{metric}_ratio'] = values[metric] / max(additive, 1e-12)
        group_rows.append(row)
    group_df = pd.DataFrame(group_rows)

    summary = {
        'seed': seed,
        'selection_seed': SELECTION_SEED,
        'device': str(device),
        'checkpoint': str(checkpoint),
        'calibration_samples': CALIB_SAMPLES,
        'test_samples': TEST_SAMPLES,
        'selected_layers': [
            {'index': int(i), 'name': layer_names[i]}
            for i in selected_layers
        ],
        'pair_indices': pair_indices,
        'group_indices': group_indices,
        'ratios': RATIOS,
        'definition': {
            'distortion': 'mean squared logit error versus dense model',
            'additive_prediction': 'sum of single-layer distortions',
            'interaction': 'joint distortion minus additive prediction',
        },
        **summarize(pair_df, 'pair'),
        **summarize(group_df, 'group'),
    }

    single_df.to_csv(output_root / 'single_layer_distortions.csv', index=False)
    pair_df.to_csv(output_root / 'pair_additivity.csv', index=False)
    group_df.to_csv(output_root / 'group_additivity.csv', index=False)
    (output_root / 'summary.json').write_text(json.dumps(summary, indent=2))

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].axhline(1.0, color='black', linewidth=1)
    axes[0].scatter(pair_df['calib_mean_sum'], pair_df['calib_mean'], alpha=0.75)
    axes[0].set_xlabel('Sum of single-layer distortions')
    axes[0].set_ylabel('Joint distortion')
    axes[0].set_title(f'Seed {seed}: pair additivity')
    axes[1].axhline(1.0, color='black', linewidth=1)
    axes[1].scatter(group_df['calib_mean_sum'], group_df['calib_mean'], alpha=0.75, color='tab:orange')
    axes[1].set_xlabel('Sum of single-layer distortions')
    axes[1].set_ylabel('Joint distortion')
    axes[1].set_title(f'Seed {seed}: group additivity')
    fig.tight_layout()
    fig.savefig(output_root / 'additivity_scatter.png', dpi=160)
    plt.close(fig)

    print(
        f'Seed {seed} finished in {time.monotonic() - started:.1f}s: '
        f'pair calibration ratio={summary["pair_calib_mean_ratio_median"]:.4f}, '
        f'group calibration ratio={summary["group_calib_mean_ratio_median"]:.4f}'
    )
    return summary

summaries = [run_seed(seed) for seed in SEEDS]
summary_df = pd.DataFrame(summaries)

metric_keys = [
    key for key in summaries[0]
    if key.endswith('ratio_median') or key.endswith('relative_interaction_median')
]
aggregate_rows = []
for key in metric_keys:
    values = [float(summary[key]) for summary in summaries]
    aggregate_rows.append({
        'metric': key,
        'mean': float(np.mean(values)),
        'std': float(np.std(values, ddof=1)),
        'seed_42': values[0],
        'seed_43': values[1],
        'seed_44': values[2],
    })
aggregate_df = pd.DataFrame(aggregate_rows)
summary_df[['seed'] + metric_keys].to_csv(AGG_ROOT / 'all_seeds_summary.csv', index=False)
aggregate_df.to_csv(AGG_ROOT / 'all_seeds_mean_std.csv', index=False)
(AGG_ROOT / 'all_seeds_summary.json').write_text(
    json.dumps({
        'seeds': SEEDS,
        'selection_seed': SELECTION_SEED,
        'calibration_samples': CALIB_SAMPLES,
        'test_samples': TEST_SAMPLES,
        'metrics': aggregate_rows,
    }, indent=2)
)

display(aggregate_df)
print('Saved aggregate results to:', AGG_ROOT)

## Optional: publish all-seed confirmation results

Set PUBLISH_RESULTS = True in the next cell after confirming the three runs completed. The publisher uploads each fixed-design seed folder and the aggregate summary.

In [ ]:
PUBLISH_RESULTS = False

if not PUBLISH_RESULTS:
    print('Publishing disabled.')
else:
    from google.colab import userdata

    token = userdata.get('GITHUB_WRITE_TOKEN')
    if not token:
        raise RuntimeError('Add GITHUB_WRITE_TOKEN with Contents read/write access.')

    repo = Path('/content/rdhp-rdp-all-seeds-upload')
    if repo.exists():
        shutil.rmtree(repo)

    askpass = Path('/tmp/github-askpass-all-seeds-write.sh')
    askpass.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        "  *Username*) printf '%s\n' x-access-token ;;\n"
        "  *Password*) printf '%s\n' \"$GITHUB_WRITE_TOKEN\" ;;\n"
        "  *) printf '\n' ;;\n"
        'esac\n'
    )
    askpass.chmod(0o700)

    env = os.environ.copy()
    env.update({
        'GIT_ASKPASS': str(askpass),
        'GIT_TERMINAL_PROMPT': '0',
        'GITHUB_WRITE_TOKEN': token,
    })

    try:
        subprocess.run([
            'git', 'clone',
            'https://github.com/hamidrg20001379/rdhp-rdp-colab-experiment.git',
            str(repo),
        ], env=env, check=True)

        for seed in SEEDS:
            source = DRIVE_ROOT / f'rdhp_rdp_runs_seed_{seed}' / 'distortion_additivity_fixed'
            if not source.exists():
                raise FileNotFoundError(f'Missing fixed-design results: {source}')
            destination = repo / 'results' / f'rdhp_rdp_runs_seed_{seed}' / 'distortion_additivity_fixed'
            shutil.copytree(source, destination, dirs_exist_ok=True)

        aggregate_destination = repo / 'results' / 'rdhp_rdp_additivity_fixed_all_seeds'
        shutil.copytree(AGG_ROOT, aggregate_destination, dirs_exist_ok=True)

        subprocess.run(['git', '-C', str(repo), 'config', 'user.name', 'Colab Results Bot'], check=True)
        subprocess.run(['git', '-C', str(repo), 'config', 'user.email', 'colab-results@users.noreply.github.com'], check=True)
        subprocess.run(['git', '-C', str(repo), 'add', 'results'], check=True)

        changed = subprocess.run([
            'git', '-C', str(repo), 'diff', '--cached', '--quiet'
        ]).returncode != 0

        if changed:
            subprocess.run([
                'git', '-C', str(repo), 'commit', '-m',
                'Publish fixed-design additivity results for seeds 42-44',
            ], check=True)
            subprocess.run([
                'git', '-C', str(repo), 'push', 'origin', 'HEAD:master',
            ], env=env, check=True)
            print('All fixed-design additivity results uploaded.')
        else:
            print('No new results to upload.')
    finally:
        askpass.unlink(missing_ok=True)